# 3.2 $\quad$ Introduction to Optimization Concepts

### Concepts

Our objective is to **minimize/maximize $f_0(x|w)$** subject to $f_i(x|w) \leq b_i$,   $\qquad i= 1,...,m$

where:

- $x = (x_1, x_2,..., x_n)$ are the variables

- $w = (w_1, w_2,..., w_n)$ are the parameters

- $f_0: \mathbb{R}^n \rightarrow \mathbb{R}$ is the objective function

- $f_i: \mathbb{R}^n \rightarrow \mathbb{R}$ are the constraints

**Solution**: $w^*$ is the parameter vector, among all vectors satisfying the constraints, that makes $f_0(x | w^*)$ attain its minimum/maximum value.

---

**EXAMPLE: PORTFOLIO OPTIMIZATION**

- *variables:* amounts or weights invested in each asset
- *parameters:* expected returns of each asset
- *constraints:* total budget, maximum/minimum investment in an asset, minimum portfolio return
- *objective:* risk or volatility

**EXAMPLE: MACHINE LEARNING – GENETIC ALGORITHM**

- *variables:* weights invested in each asset
- *parameters:* model parameters (e.g., number of generations, individuals per generation)
- *constraints:* parameter limits (e.g., maximum number of generations or maximum evolution time)
- *objective:* fitness or performance function

**EXAMPLE: DEEP LEARNING**

- *variables:* input data
- *parameters:* model parameters (e.g., connections between neurons in a neural network)
- *constraints:* parameter limits (e.g., neuronal responses can only be positive)
- *objective:* prediction error function comparing the model prediction with the expected value

There are many types of optimization problems. Portfolio optimization and machine learning are just two examples. These problems can also be very difficult to solve for several reasons:

- The problem cannot be formulated properly

- We cannot find a solution within a reasonable amount of time

- Solving the problem requires substantial computational resources

**Exceptions**: some problems can be solved efficiently and reliably:

- Linear problems using ordinary least squares (OLS)

- Convex optimization

## 1. Linear Problems Using Least Squares

This method is mainly used for model fitting, particularly in statistics and data analysis. The goal is to minimize the sum of squared differences (errors) between observed values and the values predicted by a model. It is widely used in linear regression to estimate the coefficients of a linear model.

$$\underset{w}{\min} \|Xw - y\|_2^2$$

Here, $X$ is the design matrix, containing the input features of the data. Each row $x_i$ corresponds to an observation, and each column corresponds to an independent variable. Meanwhile, $y$ is the vector of target values (the dependent variable). These are the values that we want the model to predict.

An analytical solution exists:

$w^* = (X^TX)^{-1} X^T y$

Several Python libraries can be used for linear regression:

- NumPy's polyfit

- SciPy's linregress

- **Statsmodels' OLS**

- scikit-learn's LinearRegression

The appendix to this section presents two examples of optimization using linear methods.

## 2. Convex Optimization

The following figure illustrates an optimization problem. We need to find the maxima (mountain peaks) or minima (valleys). In this problem, finding the global minimum/maximum is extremely difficult.

<div style="text-align: center;">
    <img src="../imgs/no-convexo.PNG" width="350" height="250" />
</div>

However, not all problems are this complicated. **Convex optimization** is a branch of mathematical optimization that studies how to minimize or maximize a convex function over a convex set.

<img src="../imgs/convexa.jpg" alt="CAPM" width="400"/>

The advantages of convex optimization problems include:

- **Global Optimal Solution**: Any local minimum is also a global minimum, ensuring that the solution found is the best possible.

- **Broad Applicability**: Convex optimization problems arise in many fields, including economics, engineering, machine learning, and operations research.

- **Ease of Implementation**: Algorithms for solving convex problems are straightforward to implement.

- **Computational Efficiency**: Algorithms for solving convex problems are generally efficient and well understood, allowing for fast and reliable solutions.

Optimization problems involving portfolios of financial assets are **CONVEX OPTIMIZATION PROBLEMS**:

1. **Convex Objective Function**:
   - In portfolio optimization, as in the Markowitz model, the objective is often to minimize risk (variance or standard deviation) or maximize expected return.
   - Variance (risk) is a quadratic function of asset weights and is convex by definition.

<br>

2. **Convex Constraints**:
    - Common constraints, such as requiring portfolio weights to sum to 1 and imposing non-negativity constraints, are linear (affine). These constraints are convex, meaning that the set of possible solutions (the feasible set) is convex.

For more information on convex optimization, you can study Stanford’s [Convex Optimization course](https://www.youtube.com/playlist?list=PL3940DD956CDF0622).

## Appendix: Examples of Linear Models

### Example: CAPM

A classic example of least-squares optimization in finance is linear regression used to estimate the Capital Asset Pricing Model (CAPM). Although we will study this model in detail in later topics, for now we will focus on the optimization aspect.

The CAPM is a model used to determine the expected return of an asset, taking risk and the time value of money into account. The model states that an asset's return is a linear function of the return on a broader market.

<div style="text-align: center;">
<img src="../imgs/capm.png" alt="CAPM" width="500"/>
</div>

In [ ]:
import numpy as np
import pandas as pandas
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [ ]:
tickers = ["META", "^IXIC"]
precios_df = yf.download(
    tickers, start="2018-01-01", end="2023-11-30", period="1mo", auto_adjust=True
)["Close"]
retornos_df = np.log(precios_df).diff().dropna()
retornos_df.shape

In [ ]:
# split dependent and independent variable
X = retornos_df["^IXIC"]
y = retornos_df["META"]

# Add a constant to the independent value
X1 = sm.add_constant(X)

# make regression model
model = sm.OLS(y, X1)

# fit model and print results
results = model.fit()

alpha = results.params.iloc[0]
beta = results.params.iloc[1]
print(f"Beta: {beta:.2f}")

Let us check whether this matches the beta reported by Yahoo Finance:

<img src="../imgs/beta-meta.png" alt="CAPM" width="700"/>

In the next figure, we will check how well the model fits the data. We will plot the behavior of `META` against the market, represented by the Nasdaq Composite Index (^IXIC). The figure shows how our linear model (orange line), which attempts to describe the asset return $R_i$, fits the observations (blue points) against the market return $R_m$.

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(X, y, ".")
plt.plot([X.min(), X.max()], [alpha + beta * X.min(), alpha + beta * X.max()])
plt.xlabel("Market (^IXIC)")
plt.ylabel("Asset (META)")
plt.grid()
plt.show()

When the blue points are above the model (orange line), `META` performs above the value predicted by the model. When they are below, it performs below that value. `BETA` determines the slope.

Let us apply the same procedure to a physically backed gold ETC: iShares Physical Gold GBP Hedged ETC (`IGLG.L`).

In [ ]:
tickers = ["META", "IGLG.L", "^IXIC"]
precios_df = yf.download(
    tickers, start="2018-01-01", end="2023-11-30", period="1mo", auto_adjust=True
)["Close"]
retornos_df = np.log(precios_df).diff().dropna()

In [ ]:
# split dependent and independent variable
X = retornos_df["^IXIC"]
y = retornos_df["IGLG.L"]

# Add a constant to the independent value
X1 = sm.add_constant(X)

# make regression model
model = sm.OLS(y, X1)

# fit model and print results
results = model.fit()

alpha = results.params.iloc[0]
beta = results.params.iloc[1]
print(f"Beta: {beta:.2f}")

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(X, y, ".")
plt.plot([X.min(), X.max()], [alpha + beta * X.min(), alpha + beta * X.max()])
plt.xlabel("Market (^IXIC)")
plt.ylabel("Asset (IGLG.L)")
plt.grid()
plt.show()

Comparing the two figures provides useful information:

- The orange line (the model) describes the "average" relationship between the asset and the market.

- `BETA` determines the slope of the model. A beta below one indicates a smaller response than the market, a beta of one indicates a similar response, and a beta above one indicates a larger response.

- The blue points represent the asset's behavior relative to the market. When the points are above the line, the asset performed better than predicted, and vice versa.

- The blue points lie closer to or farther from the orange line depending on the case. When they are closer, the model provides a better fit or, in other words, there is a stronger correlation between the asset and the market. When the points are farther away, the model provides a poorer fit and correlation is weaker.

In [ ]:
# Let us check the last conclusion
retornos_df.corr()

# META: 0.666128
# ^TNX: 0.178414

### Example: California Housing

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

housing = fetch_california_housing()
print(housing.DESCR)

In [ ]:
data = pd.DataFrame(housing.data, columns=housing.feature_names)
target = housing.target

In [ ]:
data

To simplify the example, we will run a linear regression with just one feature to explain house prices. We will use the median income of households, `MedInc`.

In [ ]:
X1 = sm.add_constant(data["MedInc"])
model = sm.OLS(target, X1)
results = model.fit()

alpha = results.params.iloc[0]
beta = results.params.iloc[1]
print(f"Beta: {beta:.2f}")

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(data["MedInc"], target, ".", alpha=0.1)
plt.plot(
    [data["MedInc"].min(), data["MedInc"].max()],
    [alpha + beta * data["MedInc"].min(), alpha + beta * data["MedInc"].max()],
)
plt.xlabel("Median Income (MedInc)")
plt.ylabel("Price")
plt.grid()
plt.show()

Once again, we can draw the same conclusions:

- The orange line (the model) describes the "average" relationship between median household income and house prices.

- `BETA` determines the slope of the model. A beta below one represents a smaller change in house prices relative to median income, a beta of one represents a similar change, and a beta above one represents a larger change in house prices relative to median income.

- The blue points represent house prices as a function of median income. When the points are above the line, house prices are higher than predicted, and vice versa.

- The blue points lie closer to or farther from the orange line depending on the case. When they are closer, the model fits better or, in other words, there is a stronger correlation between house prices and median household income. When they are farther away, the model fits less well.